# Musicm8 — one-click Colab training

This notebook is designed for free Colab/T4 sessions and runtime resets. **Run the big cell below once after every reset.** It mounts Drive, refreshes the GitHub code, installs compatible dependencies, restores all settings, reuses cached tokens, resumes `latest.pt` when present, trains with T4-safe defaults, and generates a sample.

> Colab must still allocate a GPU. The notebook requests one in metadata, but if CUDA is unavailable choose **Runtime → Change runtime type → GPU**, then run the same cell again.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK: SETUP → TOKENIZE → TRAIN/RESUME → SAMPLE
# Safe defaults for free Colab / Tesla T4.
# ============================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def run_cmd(cmd, *, cwd=None, log_path=None):
    """Stream subprocess output live, save it to Drive, and show a useful tail on failure."""
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd), flush=True)
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    log_f = log_path.open("w", encoding="utf-8") if log_path else None
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            if log_f:
                log_f.write(line)
                log_f.flush()
            tail.append(line.rstrip())
            tail = tail[-80:]
    finally:
        if log_f:
            log_f.close()

    rc = proc.wait()
    if rc != 0:
        print("\n❌ Command failed with exit code", rc)
        if log_path:
            print("Full log:", log_path)
        if tail:
            print("\n--- last output lines ---")
            print("\n".join(tail[-40:]))
        raise RuntimeError(f"Command failed with exit code {rc}: {' '.join(cmd)}")
    return rc

# ---------- Google Drive ----------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO_DIR = DRIVE_ROOT / "audio"
WORK_DIR = DRIVE_ROOT / "work"
LOG_DIR = WORK_DIR / "logs"
MANIFEST = DRIVE_ROOT / "manifest.jsonl"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Fresh runtime copy of GitHub code ----------
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
REPO_DIR = Path("/content/Musicm8")

if (REPO_DIR / ".git").exists():
    print("Refreshing Musicm8 from GitHub...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# ---------- Compatible dependencies ----------
# requirements.txt pins the Transformers major version so a future major release
# cannot silently break the Colab notebook.
run_cmd(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=REPO_DIR,
    log_path=LOG_DIR / "pip.log",
)

import torch

print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is connected. Choose Runtime > Change runtime type > GPU, "
        "then run THIS SAME CELL again."
    )

print("GPU:", torch.cuda.get_device_name(0))

# ---------- Persistent run settings ----------
CODEC = "encodec24"
CLIP_SECONDS = 8
STRIDE_SECONDS = 8

# T4-safe defaults. Effective batch stays 4 via gradient accumulation.
STEPS = 2000
BATCH_SIZE = 1
GRAD_ACCUM = 4
SAVE_EVERY = 50
NUM_WORKERS = 0
RUN_NAME = "tiny-overfit"
TRAIN_CONFIG = "configs/v2-colab-tiny.json"

TOKENS_DIR = WORK_DIR / f"tokens-{CODEC}"
INDEX = TOKENS_DIR / "index.jsonl"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
LATEST = RUN_DIR / "latest.pt"
SAMPLE = WORK_DIR / "sample.wav"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ WORK_DIR :", WORK_DIR)
print("✅ AUDIO_DIR:", AUDIO_DIR)
print("✅ CODEC    :", CODEC)
print("✅ RUN_NAME :", RUN_NAME)
print("✅ RUN_DIR  :", RUN_DIR)
print("✅ Config   :", TRAIN_CONFIG)
print("✅ Save every", SAVE_EVERY, "steps")

# ---------- Audio + manifest ----------
exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus"}
audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
)
print(f"✅ Audio files found: {len(audio_files)}")

if not audio_files:
    raise FileNotFoundError(f"No audio files found in {AUDIO_DIR}")

# Rebuild the simple filename-based manifest only if it is missing/empty.
if not MANIFEST.exists() or MANIFEST.stat().st_size == 0:
    with MANIFEST.open("w", encoding="utf-8") as f:
        for p in audio_files:
            caption = p.stem.replace("_", " " ).replace("-", " " )
            f.write(json.dumps(
                {"audio": str(p), "caption": f"music track, {caption}"},
                ensure_ascii=False
            ) + "\n")
    print("✅ Created manifest:", MANIFEST)
else:
    print("✅ Using existing manifest:", MANIFEST)

# ---------- Tokenize only if cache is missing ----------
if INDEX.exists() and INDEX.stat().st_size > 0:
    print("✅ Token cache already exists:", INDEX)
else:
    shutil.rmtree(TOKENS_DIR, ignore_errors=True)
    tokenize_cmd = [
        sys.executable, "-u", "tokenize_dataset.py",
        "--manifest", MANIFEST,
        "--out", TOKENS_DIR,
        "--codec", CODEC,
        "--channels", "1",
        "--clip-seconds", str(CLIP_SECONDS),
        "--stride-seconds", str(STRIDE_SECONDS),
        "--keep-tail",
        "--device", "cuda",
    ]
    print("\n🎵 Tokenizing audio...")
    run_cmd(tokenize_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "tokenize.log")

if not INDEX.exists() or INDEX.stat().st_size == 0:
    raise RuntimeError(f"No usable token index was created at {INDEX}")

print("✅ Tokenization ready:", INDEX)

# ---------- Train / resume ----------
train_cmd = [
    sys.executable, "-u", "train.py",
    "--data", INDEX,
    "--config", TRAIN_CONFIG,
    "--out", RUN_DIR,
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--steps", str(STEPS),
    "--save-every", str(SAVE_EVERY),
    "--num-workers", str(NUM_WORKERS),
    "--device", "cuda",
]

if LATEST.exists():
    train_cmd += ["--resume", LATEST]
    print("\n♻️ Resuming checkpoint:", LATEST)
else:
    print("\n🚀 Starting a fresh training run")

try:
    run_cmd(train_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "train.log")
except RuntimeError:
    # A killed/OOM subprocess often has no Python traceback. Give Colab one
    # automatic cleanup + retry before surfacing the saved log.
    print("\n⚠️ First training attempt failed. Cleaning GPU memory and retrying once...")
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    run_cmd(train_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "train_retry.log")

if not LATEST.exists():
    raise RuntimeError(f"Training ended but checkpoint was not found at {LATEST}")

print("✅ Checkpoint:", LATEST)

# ---------- Generate a sample ----------
PROMPT = "dark atmospheric electronic music with deep bass and wide synth pads"
generate_cmd = [
    sys.executable, "-u", "generate.py",
    "--checkpoint", LATEST,
    "--prompt", PROMPT,
    "--seconds", "8",
    "--seed", "42",
    "--out", SAMPLE,
    "--device", "cuda",
]

print("\n🎧 Generating sample...")
run_cmd(generate_cmd, cwd=REPO_DIR, log_path=LOG_DIR / "generate.log")

from IPython.display import Audio, display
display(Audio(str(SAMPLE)))

print("\n✅ MUSICM8 COMPLETE")
print("Checkpoint:", LATEST)
print("Sample    :", SAMPLE)
print("Logs      :", LOG_DIR)


## Optional: quick status check

Run this only if you want to inspect what is already saved in Drive without starting training.


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/Musicm8")
work = root / "work"
print("Audio folder :", root / "audio")
print("Manifest     :", root / "manifest.jsonl")
print("Token indexes:", list(work.glob("tokens-*/index.jsonl")) if work.exists() else [])
print("Checkpoints  :", list(work.glob("runs/*/latest.pt")) if work.exists() else [])
print("Samples      :", list(work.glob("*.wav")) if work.exists() else [])
print("Logs         :", list((work / "logs").glob("*.log")) if (work / "logs").exists() else [])
